# UR5e 실기 teleop

같은 입력 장치로 시뮬레이션에서 축과 버튼 방향을 확인한 뒤 실행하세요. 기본 DRY RUN은 RTDE 수신 연결로 UR의 6개 관절 각도와 TCP 자세를 읽어 가상 팔 미리보기를 엽니다. 장치 설치와 실기 준비는 [UR5e 실행 안내](../README.md)를 참고하세요.

In [ ]:
from pathlib import Path
import signal
import subprocess
import sys

cwd = Path.cwd().resolve()
repo_root = next(p for p in (cwd, *cwd.parents) if (p / 'vai_robotics/ur5e/teleop.py').is_file())

### 연결과 입력

PC와 UR 컨트롤러가 통신할 수 있는 네트워크에 연결한 뒤, 티치 펜던트에서 확인한 UR 컨트롤러의 IP를 `ROBOT_IP`에 입력하세요. `DEVICE`를 `quest` 또는 `spacemouse`로 선택하세요. Quest의 `QUEST_HOST`는 헤드셋에서 접속할 PC의 LAN 주소입니다. SpaceMouse의 `BUTTON`은 클러치로 사용할 버튼 번호입니다. 기본값은 1이며 패널에 해당 버튼의 누름·놓음 상태가 표시됩니다.

RG2를 같이 쓰려면 `GRIPPER_IP`에 OnRobot Compute Box의 주소를 지정하세요. 이 워크스테이션은 UR 컨트롤러 `192.168.0.252`, Compute Box `192.168.0.72`를 사용합니다. `GRIPPER_IP = ''`로 두면 팔만 연결합니다.

In [ ]:
ROBOT_IP = '192.168.0.252'
GRIPPER_IP = '192.168.0.72'
DEVICE = 'spacemouse'
QUEST_HOST = ''
HAND = 'right'
BUTTON = 1
ENABLE_MOTION = False

### 실행

기본값인 **DRY RUN**에서는 클러치를 누른 채 입력하면 가상 팔만 움직입니다. 클러치를 놓으면 가상 팔이 UR의 현재 관절 자세로 돌아가고, 다음에 누르면 그 자세에서 미리보기를 시작합니다.

실기를 구동할 준비가 되면 SpaceMouse의 두 버튼과 팔 클러치를 놓고 **REAL RUN · 실기 구동**을 선택한 뒤 다시 누르세요. 같은 MuJoCo 창에 UR에서 수신한 6개 관절각이 반영되므로, 클러치를 누르거나 놓고 있는 동안 모두 실제 팔의 움직임을 볼 수 있습니다. 실측 TCP와 명령 목표는 패널에서 확인하세요. 다시 **DRY RUN**을 선택하면 실기를 정지하고 같은 창에서 미리보기를 재개합니다. 세션을 끝내려면 패널을 닫거나 실행 중인 셀을 중단하세요.

SpaceMouse의 **0번과 1번을 함께 누르면 RG2가 닫히고, 하나라도 놓으면 열립니다**. 두 버튼을 계속 누른 채 팔을 움직여 물체를 옮길 수 있습니다. DRY RUN에서는 가상 그리퍼의 개도가 초당 40 mm씩 변합니다. 닫히는 중간에 버튼 하나를 놓으면 그 위치에서 다시 열립니다. REAL RUN으로 전환한 뒤 두 버튼을 새로 함께 누르면 Compute Box로 구동 명령이 전달됩니다. REAL RUN의 손가락은 수신한 실측 개도를 따릅니다.

In [ ]:
if not ROBOT_IP:
    raise ValueError('ROBOT_IP에 UR 컨트롤러 주소를 입력하세요.')
args = ['--backend', 'real', '--input', DEVICE, '--robot-ip', ROBOT_IP]
if GRIPPER_IP:
    args += ['--gripper-ip', GRIPPER_IP]
if DEVICE == 'quest':
    if not QUEST_HOST:
        raise ValueError('QUEST_HOST에 PC의 LAN 주소를 입력하세요.')
    args += ['--quest-host', QUEST_HOST, '--hand', HAND]
elif DEVICE == 'spacemouse':
    args += ['--button', str(BUTTON)]
if ENABLE_MOTION:
    args += ['--enable-motion']
command = [sys.executable, '-u', '-m', 'vai_robotics.ur5e.teleop', *args]
with subprocess.Popen(command, cwd=repo_root, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        exit_code = process.wait()
        if exit_code not in (0, -signal.SIGINT, 128 + signal.SIGINT):
            raise subprocess.CalledProcessError(exit_code, command)
    except KeyboardInterrupt:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.terminate()